In [10]:
!pip install requests ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 40.4 MB/s eta 0:00:00


In [11]:
import requests
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Weather descriptions
weather_codes = {
    0: "☀️ Clear Sky",
    1: "🌤️ Mainly Clear",
    2: "⛅ Partly Cloudy",
    3: "☁️ Overcast",
    45: "🌫️ Fog",
    48: "🌫️ Fog",
    51: "🌦️ Light Drizzle",
    53: "🌦️ Moderate Drizzle",
    55: "🌧️ Dense Drizzle",
    61: "🌧️ Slight Rain",
    63: "🌧️ Moderate Rain",
    65: "🌧️ Heavy Rain",
    71: "🌨️ Slight Snow",
    73: "❄️ Moderate Snow",
    75: "❄️ Heavy Snow",
    80: "🌦️ Rain Showers",
    81: "🌧️ Rain Showers",
    82: "⛈️ Heavy Rain Showers",
    95: "⛈️ Thunderstorm",
    96: "⛈️ Thunderstorm + Hail",
    99: "⛈️ Heavy Thunderstorm"
}

# -----------------------------
# Create GUI
# -----------------------------

title = widgets.HTML(
    "<h1>🌦️ Weather App</h1>"
)

city_box = widgets.Text(
    placeholder="Enter city name",
    description="City:",
    layout=widgets.Layout(width="400px")
)

search_button = widgets.Button(
    description="🔍 Get Weather",
    button_style="primary",
    layout=widgets.Layout(width="150px")
)

output = widgets.Output()

# -----------------------------
# Weather Function
# -----------------------------

def get_weather(city):

    # Step 1: Find city coordinates
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = requests.get(
        geo_url,
        params=geo_params
    )

    if response.status_code != 200:
        return None

    geo_data = response.json()

    if "results" not in geo_data:
        return None

    location = geo_data["results"][0]

    latitude = location["latitude"]
    longitude = location["longitude"]

    city_name = location["name"]
    country = location.get("country", "")

    # Step 2: Get weather
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "apparent_temperature,"
            "weather_code,"
            "wind_speed_10m,"
            "precipitation"
        ),
        "timezone": "auto"
    }

    weather_response = requests.get(
        weather_url,
        params=weather_params
    )

    if weather_response.status_code != 200:
        return None

    weather_data = weather_response.json()

    current = weather_data["current"]

    return {
        "city": city_name,
        "country": country,
        "temperature": current["temperature_2m"],
        "humidity": current["relative_humidity_2m"],
        "feels_like": current["apparent_temperature"],
        "wind": current["wind_speed_10m"],
        "rain": current["precipitation"],
        "condition": weather_codes.get(
            current["weather_code"],
            "Unknown"
        )
    }


# -----------------------------
# Button Function
# -----------------------------

def search_weather(button):

    with output:

        clear_output(wait=True)

        city = city_box.value.strip()

        if city == "":
            display(HTML(
                "<h3>⚠️ Please enter a city name.</h3>"
            ))
            return

        try:

            weather = get_weather(city)

            if weather is None:
                display(HTML(
                    "<h3>❌ City not found. "
                    "Please try again.</h3>"
                ))
                return

            # Weather card
            html = f"""
            <div style="
                width: 420px;
                padding: 25px;
                margin-top: 20px;
                border-radius: 20px;
                background: linear-gradient(
                    135deg,
                    #2193b0,
                    #6dd5ed
                );
                color: white;
                font-family: Arial;
                box-shadow: 0px 5px 20px rgba(0,0,0,0.2);
            ">

                <h2 style="text-align:center;">
                    🌦️ Weather Information
                </h2>

                <h3 style="text-align:center;">
                    📍 {weather["city"]},
                    {weather["country"]}
                </h3>

                <hr>

                <h1 style="text-align:center;">
                    {weather["temperature"]} °C
                </h1>

                <h3 style="text-align:center;">
                    {weather["condition"]}
                </h3>

                <hr>

                <p>🌡️ Feels Like:
                {weather["feels_like"]} °C</p>

                <p>💧 Humidity:
                {weather["humidity"]}%</p>

                <p>💨 Wind Speed:
                {weather["wind"]} km/h</p>

                <p>🌧️ Precipitation:
                {weather["rain"]} mm</p>

            </div>
            """

            display(HTML(html))

        except Exception as e:

            display(HTML(
                f"<h3>❌ Error: {e}</h3>"
            ))


# Connect button
search_button.on_click(search_weather)


# Display application
display(
    title,
    city_box,
    search_button,
    output
)

HTML(value='<h1>🌦️ Weather App</h1>')

Text(value='', description='City:', layout=Layout(width='400px'), placeholder='Enter city name')

Button(button_style='primary', description='🔍 Get Weather', layout=Layout(width='150px'), style=ButtonStyle())

Output()